# Extracting Column Data from IFC for Capacity Checks

**Level 1 tutorial · Role: Analyst · Focus area: Structural Analysis · BIM use: 08 Structural Analysis**
Group 24 · Gulshan Kumar, Aida Amirbabaei · Course 41934 Advanced BIM (DTU)

## Summary
This beginner tutorial shows structural analysts how to load an IFC model with IfcOpenShell, extract column dimensions, materials and storeys, and combine them with external load data in a pandas table for later capacity checks.

## Why this tutorial?
Our tool idea (A2e) checks whether existing columns can carry the extra load from a vertical extension. Its first three steps are:

1. Load the IFC model with IfcOpenShell
2. Extract column geometry and material properties
3. Import existing and additional loads

Those three steps are what you will practise here. Nothing in this notebook needs prior Python knowledge.

## Learning objectives
After this tutorial you can:
1. Open an IFC file with IfcOpenShell and read its schema, units and number of columns.
2. Extract for every `IfcColumn`: GlobalId, mark, storey, material, cross-section and length.
3. Handle columns whose geometry is *not* a simple rectangle (a real problem in this model).
4. Combine the IFC data with an external load table and detect columns with missing load data.

## Before you start
* Python 3.10+ and Jupyter
* `pip install ifcopenshell pandas`
* The model **`26-08-STR.ifc`** (course structural model, Revit 2027 export, IFC4X3). Put it in the `data/` folder.
* Time needed: about 45–60 minutes

> **Important:** `data/loads_example.csv` contains **invented example numbers**. They are only there to practise merging data. They are not real loads.

## Step 0 · Imports

In [ ]:
from collections import Counter

import ifcopenshell
import ifcopenshell.util.element as el_util
import ifcopenshell.util.unit as unit_util
import pandas as pd

IFC_PATH = "data/26-08-STR.ifc"

---
# Step 1 · Load the IFC model and read its units

`ifcopenshell.open` reads the file. Always check the **units** first: IFC files can store lengths in metres, millimetres or something else, and mixing them up gives wrong results.

In [ ]:
model = ifcopenshell.open(IFC_PATH)
print("Schema:", model.schema)

# Factor that converts the project length unit to metres
unit_scale = unit_util.calculate_unit_scale(model)
print("Length unit -> metres:", unit_scale)

# All units defined in the file
for unit in model.by_type("IfcSIUnit"):
    print(unit.UnitType, unit.Prefix, unit.Name)

**What to look for:** this model stores lengths in **millimetres** (`unit_scale = 0.001`) but areas in **square metres**. Remember this for Step 2: a length of `3670` means 3670 mm, but an area of `0.104` means 0.104 m².

In [ ]:
columns = model.by_type("IfcColumn")
print("Number of IfcColumn:", len(columns))

for column in columns[:3]:
    print(column.GlobalId, "|", column.Name)

### How are columns named in this model?
The `Name` follows the Revit pattern **`Family : Type : RevitID`**. The *Type* part is the structural **mark** (for example `S114`), which is what the structural report uses.

* `M_Concrete-Rectangular-Column` → existing concrete columns
* `Concrete-Rectangular-Column-Jacketing` → new concrete jackets around existing columns (`.J` marks)
* `Glulam-Square-Column-GL30h` → new timber columns (`.T` marks)

In [ ]:
def split_name(column):
    """'Family:Type:ID' -> (family, mark)."""
    parts = (column.Name or "").split(":")
    family = parts[0]
    mark = parts[1] if len(parts) > 1 else None
    return family, mark

print(Counter(split_name(c)[0] for c in columns))

**Your turn 1:** count how many existing concrete columns there are per mark. *Hint: use `Counter` and `split_name`, and keep only the family `M_Concrete-Rectangular-Column`.*

In [ ]:
# Your code here

---
# Step 2 · Extract geometry, storey and material

| What we need | Where it is in IFC | IfcOpenShell function |
|---|---|---|
| Storey | `IfcRelContainedInSpatialStructure` | `el_util.get_container(col, ifc_class="IfcBuildingStorey")` |
| Material | `IfcRelAssociatesMaterial` | `el_util.get_materials(col)` |
| Cross-section | `IfcRectangleProfileDef` inside the *type's* geometry | `el_util.get_type(col)` → `RepresentationMaps` |
| Length and area | quantity set `Qto_ColumnBaseQuantities` | `el_util.get_psets(col, qtos_only=True)` |

Why two sources for the cross-section? In this model the geometry is stored once on the column *type* and re-used. Most columns have a simple `IfcRectangleProfileDef`, but some (mainly the jackets) are free-form shapes. For those we fall back to the quantity set.

In [ ]:
def get_rectangle_profile(column):
    """Return the IfcRectangleProfileDef of a column's type, or None."""
    column_type = el_util.get_type(column)
    if column_type is None:
        return None
    for rep_map in column_type.RepresentationMaps or []:
        for item in rep_map.MappedRepresentation.Items:
            if item.is_a("IfcExtrudedAreaSolid") and item.SweptArea.is_a("IfcRectangleProfileDef"):
                return item.SweptArea
    return None


def get_storey_name(column):
    storey = el_util.get_container(column, ifc_class="IfcBuildingStorey")
    return storey.Name if storey else None


def get_material_names(column):
    materials = el_util.get_materials(column)
    return "; ".join(sorted({m.Name for m in materials if m is not None and m.Name})) or None


def classify(family):
    if family.startswith("M_Concrete"):
        return "Existing concrete"
    if "Jacketing" in family:
        return "New jacket"
    if "Glulam" in family:
        return "New timber"
    return "Other"

In [ ]:
rows = []
for column in columns:
    family, mark = split_name(column)
    qto = el_util.get_psets(column, qtos_only=True).get("Qto_ColumnBaseQuantities", {})

    length = qto.get("Length")        # project length unit (mm here)
    area = qto.get("CrossSectionArea")  # square metres in this model

    profile = get_rectangle_profile(column)
    if profile is not None:
        # sorted() -> b is always the smaller side, h the larger side
        b_mm, h_mm = sorted([profile.XDim * unit_scale * 1000, profile.YDim * unit_scale * 1000])
        dim_source = "profile"
    else:
        b_mm = h_mm = None
        dim_source = "quantity only"

    rows.append({
        "GlobalId": column.GlobalId,
        "Name": column.Name,
        "Class": classify(family),
        "Mark": mark,
        "Storey": get_storey_name(column),
        "Material": get_material_names(column),
        "b_mm": round(b_mm) if b_mm is not None else None,
        "h_mm": round(h_mm) if h_mm is not None else None,
        "Length_m": round(length * unit_scale, 3) if length is not None else None,
        "Area_m2": round(area, 4) if area is not None else None,
        "Dim_source": dim_source,
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
print(df.groupby(["Class", "Storey"]).size())
print()
print(df["Dim_source"].value_counts())

**Read the result.** Not every column has a rectangular profile. Those with `Dim_source = "quantity only"` still have `Area_m2` and `Length_m`, which is enough for an axial capacity estimate, but **not** for checking `b × h` or for bending.

### Sanity check: do profile and quantity agree?
If the units are right, `b × h` (in m²) must be almost equal to `Area_m2`. A mismatch is a warning that you mixed up units or that the geometry is not a plain rectangle. Expect to find a few columns with a large difference. Part of the exercise is finding out why.

In [ ]:
rect = df[df["Dim_source"] == "profile"].copy()
rect["Area_from_dims_m2"] = rect["b_mm"] * rect["h_mm"] / 1e6
rect["Diff_m2"] = (rect["Area_from_dims_m2"] - rect["Area_m2"]).abs()
print("Largest difference (m2):", rect["Diff_m2"].max())
rect.sort_values("Diff_m2", ascending=False).head(5)[["Name", "b_mm", "h_mm", "Area_m2", "Area_from_dims_m2"]]

### Compare the model with the structural report
The report (Team 08 structural report, Table 6, page 19) lists the existing column sizes. We typed the *existing* columns into `data/report_table6_existing_columns.csv`. Now we check what the IFC says for the same marks.

In [ ]:
report = pd.read_csv("data/report_table6_existing_columns.csv")
report["b_report"] = report[["Report_dim_1_mm", "Report_dim_2_mm"]].min(axis=1)
report["h_report"] = report[["Report_dim_1_mm", "Report_dim_2_mm"]].max(axis=1)

existing = df[(df["Class"] == "Existing concrete") & (df["Dim_source"] == "profile")]

# For every mark, collect the (b, h) pairs found in the IFC model
sizes = {}
for _, row in existing.iterrows():
    sizes.setdefault(row["Mark"], set()).add((row["b_mm"], row["h_mm"]))

check = report.copy()
check["IFC_sizes"] = check["Mark"].map(lambda m: sorted(sizes.get(m, [])))
check["Match"] = check.apply(
    lambda r: any(abs(b - r["b_report"]) < 1 and abs(h - r["h_report"]) < 1
                  for b, h in r["IFC_sizes"]),
    axis=1,
)
check[["Mark", "b_report", "h_report", "IFC_sizes", "Match"]]

**Your turn 2:** which marks do *not* match? Look at them in a viewer (Bonsai or any IFC viewer). Is the report wrong, the model wrong, or is the table simply an older state of the design? Also look at marks that appear in the model but not in the report table.

In [ ]:
# Your code here

---
# Step 3 · Import existing and additional loads

Loads are **not stored in this IFC model**, so they must come from outside, usually a spreadsheet exported from the analysis software. We read them from a CSV file and join them to the IFC data using the column **mark**.

> `data/loads_example.csv` contains invented numbers. Replace it with your own data in a real project.

In [ ]:
loads = pd.read_csv("data/loads_example.csv")
loads.head()

In [ ]:
existing_cols = df[df["Class"] == "Existing concrete"]

data = existing_cols.merge(loads, on="Mark", how="left", indicator=True)

missing = data[data["_merge"] == "left_only"]
print("Existing columns:", len(data))
print("Columns WITHOUT load data:", len(missing))
print("Marks without load data:", sorted(missing["Mark"].dropna().unique()))

**Why this check matters:** a `left` merge never throws an error when data is missing. It silently leaves empty cells, and a later calculation would simply ignore those columns. Always count the gaps.

Finally, save the combined table. This file is the input to the next steps of the tool (load combinations and axial resistance), which are *not* part of this tutorial.

In [ ]:
data = data.drop(columns="_merge")
data.to_csv("column_data.csv", index=False)
data[["Mark", "Storey", "b_mm", "h_mm", "Length_m", "Existing_Load_kN",
      "Additional_Dead_kN", "Additional_Imposed_kN"]].head(10)

**Your turn 3:** add a column `Additional_Total_kN = Additional_Dead_kN + Additional_Imposed_kN` and list the five existing columns with the largest value. Why would summing dead and imposed load like this be *wrong* in a real design check?

---
## Common problems
| Problem | Cause | Fix |
|---|---|---|
| Dimensions are 1000× too large or small | Length unit not converted | Multiply by `unit_scale` |
| Area looks wrong | Area unit is not the length unit squared | Check `IfcSIUnit` list |
| `None` for dimensions | Geometry is not a rectangle profile | Use the quantity set (`Area_m2`) |
| `None` for `Length_m` | Some columns have no `Length` quantity | Take the length from the storey heights or the report |
| Mark not found by `Name` | Different naming convention in another model | Print a few names first, then adapt `split_name` |
| Many empty load cells | Marks differ between report and model | Check the `missing` list above |

## What comes next (not in this tutorial)
Load combinations with partial safety factors, axial resistance, utilization ratio and the assessment report.

## Peer feedback (please answer after trying the tutorial)
1. Did the notebook run without errors? Which step failed, and what was the error message?
2. Which step was hardest to follow?
3. Were the learning objectives achieved? (1–5)
4. Was the Level 1 (beginner) assumption right for you?
5. One thing we should add, and one we should remove.